# KNN 房屋型別分類
輸入「總層數」與「單價」，用 KNN 分類預測房屋型別（構造-類別）。

In [ ]:
# 1. Clone 專案
import os
if not os.path.exists('DataMiningExercises'):
    !git clone https://github.com/JackPan0521/DataMiningExercises.git
else:
    !git -C DataMiningExercises pull
print('Done')

In [ ]:
# 2. 安裝套件（Colab 通常已內建）
# !pip install scikit-learn pandas numpy

In [ ]:
# 3. Imports
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
# 4. 參數設定
CSV_PATH   = 'DataMiningExercises/臺北市房屋構造標準單價表/臺北市房屋構造標準單價表-35層以下(112年7月起適用)-revised.csv'
K_MIN      = 1
K_MAX      = 20
CV_SPLITS  = 5
REPORT_CSV = 'knn_classification_k_report.csv'

# ★ 在這裡輸入要預測的新資料
INPUT_FLOOR = 12      # 總層數
INPUT_PRICE = 15230   # 單價

In [ ]:
# 5. 載入 CSV
last_error = None
for enc in ['utf-8-sig', 'cp950', 'big5', 'utf-8']:
    try:
        raw_df = pd.read_csv(CSV_PATH, encoding=enc)
        print(f'Loaded with encoding: {enc}, rows={len(raw_df)}')
        break
    except Exception as exc:
        last_error = exc
else:
    raise RuntimeError(f'Failed to read CSV; last error: {last_error}')
raw_df.head(3)

In [ ]:
# 6. 轉成長格式
value_cols = [c for c in raw_df.columns if c != '總層數']
long_df = raw_df.melt(id_vars=['總層數'], value_vars=value_cols, var_name='型別', value_name='單價')
long_df['單價'] = pd.to_numeric(long_df['單價'], errors='coerce')
long_df = long_df.dropna(subset=['單價']).copy()
print(f'Long format rows: {len(long_df)}')
long_df.head(3)

In [ ]:
# 7. K 值搜尋（CV Accuracy）
if K_MIN < 1 or K_MAX < K_MIN:
    raise ValueError('Invalid k range')

X = long_df[['總層數', '單價']]
y = long_df['型別']
cv = KFold(n_splits=CV_SPLITS, shuffle=True, random_state=42)

rows = []
for k in range(K_MIN, K_MAX + 1):
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=k, weights='distance'))
    ])
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    rows.append({'k': k, 'cv_accuracy_mean': scores.mean(), 'cv_accuracy_std': scores.std(ddof=1)})
    print(f'  k={k:2d}  CV Accuracy={scores.mean():.4f} ± {scores.std(ddof=1):.4f}')

k_report = pd.DataFrame(rows)
best_k = int(k_report.loc[k_report['cv_accuracy_mean'].idxmax(), 'k'])
k_report.to_csv(REPORT_CSV, index=False, encoding='utf-8-sig')
print(f'\nBest k by CV accuracy: {best_k}')
print(f'Saved: {REPORT_CSV}')
k_report.sort_values('cv_accuracy_mean', ascending=False).head(5)

In [ ]:
# 8. 訓練最終分類器
try:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
except ValueError:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print('Warning: stratified split not possible; fallback to random split.')

clf = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=best_k, weights='distance'))
])
clf.fit(X_train, y_train)

test_acc = accuracy_score(y_test, clf.predict(X_test))
print(f'Training rows: {len(X_train)}, Test rows: {len(X_test)}')
print(f'Holdout test accuracy: {test_acc:.4f}')

In [ ]:
# 9. 預測新資料
new_x = pd.DataFrame({'總層數': [INPUT_FLOOR], '單價': [INPUT_PRICE]})

pred_label = clf.predict(new_x)[0]
pred_prob  = clf.predict_proba(new_x)[0]
classes    = clf.named_steps['knn'].classes_
top_idx    = np.argsort(pred_prob)[::-1][:3]

print('=== KNN Classification Summary ===')
print(f'Best k by CV accuracy : {best_k}')
print(f'Holdout test accuracy : {test_acc:.4f}')
print()
print('=== New Input Prediction ===')
print(f'Input: 總層數={INPUT_FLOOR}, 單價={INPUT_PRICE}')
print(f'Predicted 型別: {pred_label}')
print('Top-3 probabilities:')
for i in top_idx:
    print(f'  {classes[i]}: {pred_prob[i]:.4f}')

In [ ]:
# 10. 下載報告（選用）
try:
    from google.colab import files
    files.download(REPORT_CSV)
except ImportError:
    print('非 Colab 環境，略過下載步驟')